# MLB Phase 1C — Pre-game model training and reproducibility audit

**Purpose:** Train the revised **22-feature** pre-game model on the cleaned 2025 regular season sample, using a strictly chronological train/validation/test split. The notebook reports DNN and baseline performance, probability calibration, and reproducible saved predictions. It **does not** retrain the In-game model or modify the web dashboard.

**Input:** `phase1b_output.zip` from Phase 1B. Upload it in the second code cell.

**Important:** The Phase 1B file already computes features from strictly earlier dates. In this notebook the missing starting-pitcher statistics are recomputed with **train-set-only medians**, replacing the provisional diagnostic defaults. An early-season 0.5 win-percentage policy remains in effect and is recorded in the audit.


In [ ]:
import sys, subprocess, importlib.util
for pkg in ['tensorflow','scikit-learn','pandas','matplotlib']:
    if importlib.util.find_spec(pkg.replace('-','_')) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import os, json, zipfile, io, random, warnings
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
from google.colab import files
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve
print('TensorFlow:',tf.__version__,'sklearn available')

In [ ]:
uploaded=files.upload()
assert len(uploaded)==1 and next(iter(uploaded)).endswith('.zip'),'Upload only phase1b_output.zip'
source=next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[source])) as z:
    df=pd.read_csv(z.open('pregame_asof_prior_date_DIAGNOSTIC.csv'))
    audit=json.load(z.open('audit_report.json'))
assert len(df)==2430 and df.gamePk.nunique()==2430
print('Input:',len(df),'games, range',df.date.min(),df.date.max())
print('Original audit:',audit)

## 1. Chronological partitions and train-only preprocessing

All games played on the same calendar date belong to the same partition. The last 15% of the **distinct dates** are used as the test period; the preceding 15% as validation. This is a genuine future-held-out evaluation, not a randomized game split. All reported model selection and imputation use training/validation data only; test labels are never used in fitting.


In [ ]:
SEED=2026
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
df['date']=pd.to_datetime(df['date']); df=df.sort_values(['date','gamePk']).reset_index(drop=True)
dates=np.array(sorted(df.date.unique()))
a=int(len(dates)*0.70); b=int(len(dates)*0.85)
assert 0<a<b<len(dates)
train_dates=set(dates[:a]); val_dates=set(dates[a:b]); test_dates=set(dates[b:])
df['split']=np.where(df.date.isin(train_dates),'train',np.where(df.date.isin(val_dates),'validation','test'))
print(df.groupby('split',sort=False).agg(games=('gamePk','size'),start=('date','min'),end=('date','max'),home_wins=('target','sum')).to_string())
assert df.groupby('date').split.nunique().max()==1
assert set(df[df.split=='train'].gamePk).isdisjoint(set(df[df.split=='test'].gamePk))
FEATURES=audit['feature_names']; assert len(FEATURES)==22
trainmask=df.split=='train';valmask=df.split=='validation';testmask=df.split=='test'
# Do not use the arbitrary diagnostic constants for pitcher stats without a prior start.
# Instead, source-level missingness is restored before fitting train-only medians.
missing_counts={}
for side in ['home','away']:
    is_missing=df[f'{side}_sp_source']!='prior_game_logs'
    for stat in ['era','whip','k9']:
        col=f'{side}_sp_{stat}'
        df.loc[is_missing,col]=np.nan
        missing_counts[col]=int(df[col].isna().sum())
# Train-only medians from BOTH home and away observed pitcher values for a common prior.
medians={}
for stat in ['era','whip','k9']:
    cols=[f'home_sp_{stat}',f'away_sp_{stat}']
    prior=pd.concat([df.loc[trainmask,c].dropna() for c in cols],ignore_index=True)
    med=float(prior.median()); medians[stat]=med
    for c in cols: df[c]=df[c].fillna(med)
# Recalculate differences *after* replacing missing pitcher values.
for stat in ['era','whip','k9']:
    df[f'sp_{stat}_diff']=df[f'home_sp_{stat}']-df[f'away_sp_{stat}']
assert df[FEATURES].notna().sum().sum()==len(df)*len(FEATURES)
print('Train-only pitcher medians:',medians)
print('Pitcher missing counts:',missing_counts)
print('Early-season win_pct defaults (either team):',int((df.home_winpct_imputed|df.away_winpct_imputed).sum()))

## 2. Fit DNN and baselines

The DNN keeps the manuscript's 128→64→32 hidden-layer structure and applies Dropout after the first and second hidden layers (0.3, 0.3) and L2 regularization. The scaler is fitted exclusively on training rows. The validation set is used for early stopping and model selection; the test set is scored once after training.


In [ ]:
Xraw=df[FEATURES].to_numpy(dtype='float64');y=df.target.to_numpy(dtype='int32')
scaler=StandardScaler().fit(Xraw[trainmask])
X=scaler.transform(Xraw).astype('float32')
Xt,Xv,Xe=X[trainmask],X[valmask],X[testmask]
yt,yv,ye=y[trainmask],y[valmask],y[testmask]
from tensorflow.keras import regularizers
inp=tf.keras.Input(shape=(len(FEATURES),))
x=tf.keras.layers.Dense(128,activation='relu',kernel_regularizer=regularizers.l2(0.001))(inp)
x=tf.keras.layers.Dropout(0.3)(x)
x=tf.keras.layers.Dense(64,activation='relu',kernel_regularizer=regularizers.l2(0.001))(x)
x=tf.keras.layers.Dropout(0.3)(x)
x=tf.keras.layers.Dense(32,activation='relu',kernel_regularizer=regularizers.l2(0.001))(x)
out=tf.keras.layers.Dense(1,activation='sigmoid')(x)
model=tf.keras.Model(inp,out)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss='binary_crossentropy',metrics=['accuracy'])
history=model.fit(Xt,yt,validation_data=(Xv,yv),epochs=200,batch_size=64,verbose=0,
                  callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True)])
print('epochs completed',len(history.history['loss']),'best epoch',int(np.argmin(history.history['val_loss'])+1))
print('DNN parameters:',model.count_params())
models={
 'Logistic regression':LogisticRegression(max_iter=3000,random_state=SEED),
 'Random forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=5,random_state=SEED,n_jobs=-1),
 'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=150,learning_rate=0.05,l2_regularization=1.0,random_state=SEED)
}
for name,m in models.items():m.fit(Xt,yt)
print('Baseline fits complete')

In [ ]:
def calc_metrics(yy,p):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    return {'accuracy':float(accuracy_score(yy,p>=0.5)),
            'auc':float(roc_auc_score(yy,p)),
            'brier':float(brier_score_loss(yy,p)),
            'log_loss':float(log_loss(yy,p,labels=[0,1]))}
def ece_score(yy,p,n_bins=10):
    edges=np.linspace(0,1,n_bins+1); idx=np.minimum(np.searchsorted(edges,p,side='right')-1,n_bins-1)
    s=0
    for i in range(n_bins):
        mask=idx==i
        if mask.sum()>0: s+=mask.mean()*abs(yy[mask].mean()-p[mask].mean())
    return float(s)
pred={'DNN':model.predict(Xe,verbose=0).reshape(-1)}
for name,m in models.items():pred[name]=m.predict_proba(Xe)[:,1]
rows=[]
for name,p in pred.items():
    row={'model':name,**calc_metrics(ye,p),'ECE_10_bins':ece_score(ye,p)};rows.append(row)
metrics_df=pd.DataFrame(rows)
print(metrics_df.to_string(index=False,float_format=lambda z:f'{z:.4f}'))
# Bootstrap paired accuracy-difference confidence interval on held-out games
rng=np.random.default_rng(SEED);boot=[]
if len(ye):
    for _ in range(1000):
        ix=rng.integers(0,len(ye),len(ye))
        boot.append(np.mean((pred['DNN'][ix]>=0.5)==ye[ix])-np.mean((pred['Logistic regression'][ix]>=0.5)==ye[ix]))
print('DNN minus logistic accuracy difference and 95% paired bootstrap CI:',
      np.mean((pred['DNN']>=.5)==ye)-np.mean((pred['Logistic regression']>=.5)==ye),np.quantile(boot,[.025,.975]))
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(history.history['loss'],label='train');axs[0].plot(history.history['val_loss'],label='validation')
axs[0].legend();axs[0].set_title('DNN loss');axs[0].set_xlabel('Epoch')
for name,p in pred.items():
    obs,mean=calibration_curve(ye,p,n_bins=10,strategy='quantile')
    axs[1].plot(mean,obs,marker='o',label=name)
axs[1].plot([0,1],[0,1],'--',color='gray');axs[1].legend(fontsize=8)
axs[1].set_title('Test-set probability calibration');axs[1].set_xlabel('Mean predicted probability');axs[1].set_ylabel('Observed win fraction')
plt.tight_layout();plt.savefig('pregame_phase1c_figures.png',dpi=160);plt.show()

## 3. Export evidence package

The ZIP contains a test-predictions table, metrics and a technical audit, reproducible model JSON (network weights and scaler), a Keras model, partitions, and calibration/loss plots. **Do not substitute this new model into the dashboard yet**: JavaScript feature generation must be reconciled first.


In [ ]:
outdir=Path('phase1c_output');outdir.mkdir(exist_ok=True)
model.save(outdir/'pregame_model.keras')
metrics_df.to_csv(outdir/'pregame_metrics.csv',index=False)
df[['gamePk','date','split','target']].to_csv(outdir/'game_partitions.csv',index=False)
test_pred=df.loc[testmask,['gamePk','date','target']].reset_index(drop=True)
for name,p in pred.items():test_pred['prob_'+name.lower().replace(' ','_')]=p
test_pred.to_csv(outdir/'pregame_test_predictions.csv',index=False)
df.to_csv(outdir/'pregame_features_cleaned_2430.csv',index=False)
import shutil
shutil.copy('pregame_phase1c_figures.png',outdir/'pregame_phase1c_figures.png')
weights={'features':FEATURES,'scaler':{'mean':scaler.mean_.tolist(),'scale':scaler.scale_.tolist()},
         'layers':[{'W':layer.get_weights()[0].tolist(),'b':layer.get_weights()[1].tolist(),
                    'activation':layer.activation.__name__} for layer in model.layers if isinstance(layer,tf.keras.layers.Dense)],
         'note':'NO clipping applied; test metrics computed on raw sigmoid probabilities'}
with open(outdir/'pregame_model_export.json','w') as f:json.dump(weights,f)
report={'source_audit':audit,'chronological_split':{
  split:{'games':int(mask.sum()),'start':str(df.loc[mask,'date'].min().date()),'end':str(df.loc[mask,'date'].max().date())}
  for split,mask in [('train',trainmask),('validation',valmask),('test',testmask)]},
  'imputation':{'pitcher_train_only_medians':medians,'pitcher_missing_counts':missing_counts,
     'early_win_pct_imputed_games':int((df.home_winpct_imputed|df.away_winpct_imputed).sum())},
  'model':{'architecture':[22,128,64,32,1],'epochs_completed':len(history.history['loss']),
           'best_epoch':int(np.argmin(history.history['val_loss'])+1),'parameters':model.count_params()},
  'accuracy_diff_DNN_minus_logistic':float(np.mean((pred['DNN']>=.5)==ye)-np.mean((pred['Logistic regression']>=.5)==ye)),
  'paired_bootstrap_95ci':np.quantile(boot,[.025,.975]).tolist(),
  'limitations':['Pitcher IDs may reflect official recorded starters rather than pregame announcements',
                 'The neutral early-win-percentage policy is retained from Phase 1B',
                 'In-game model and JavaScript runtime not yet audited in this step']}
with open(outdir/'training_audit.json','w') as f:json.dump(report,f,indent=2,ensure_ascii=False)
with zipfile.ZipFile('phase1c_output.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in outdir.iterdir(): z.write(path,arcname=path.name)
print('Files exported:',[p.name for p in outdir.iterdir()]);files.download('phase1c_output.zip')